# Instructor Notebook 06 — LLMs: The Real System
**ComplianceGPT Lab · REU 2026 · Week 2**

> **Teaching script:** Run every cell top-to-bottom. Uses only pandas + sklearn — no model inference required.
> Everything in this notebook comes from the actual Gemma3-4B experiment results.

**Learning arc:**
1. What an LLM actually produces on HIPAA scenarios (real outputs)
2. 94.2% accuracy — what that means and where the 8 errors are
3. Error anatomy: FP vs FN, oracle hallucinations, under-extraction
4. Reading the extracted JSON: CI 5-tuple + 24 oracle predicates
5. The formal engine: how Soufflé turns predicates into verdicts
6. Research directions: how would YOU improve this?

In [1]:
import pandas as pd
import numpy as np
import json
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

HIPAA_PATH = '/Users/priscilladanso/Documents/GitHub/COMPLIANCEGPT/experiments/finalserverrun/final_vast_gemma3_4b.csv'
df = pd.read_csv(HIPAA_PATH)

print(f'Loaded Gemma3-4B experiment results')
print(f'  Rows:    {len(df)} HIPAA court cases')
print(f'  Columns: {list(df.columns)}')

Loaded Gemma3-4B experiment results
  Rows:    137 HIPAA court cases
  Columns: ['row_id', 'format', 'topic', 'complexity', 'regulation', 'question', 'ground_truth', 'strategy', 'model', 'verdict', 'verdict_norm', 'match', 'citations', 'latency_s', 'llm_answer', 'explanation', 'scenario_json', 'souffle_facts', 'llm_raw', 'error', 'timestamp']


---
## Part 1 · System Overview — The Pipeline You're Researching

> **Say:** "This is the actual pipeline. We ran Gemma3-4B on a V100 GPU on the Stony Brook cluster. It processed 137 court cases. Let's see what it produced."

```
Court case text
    ↓  (prompt + schema)
 Gemma3-4B on GPU
    ↓  (extracts JSON: CI 5-tuple + 24 oracle predicates)
 Soufflé Datalog engine
    ↓  (applies HIPAA rules formally)
 PERMITTED or DENIED
    ↓  (compare to ground truth)
 Match Y/N  → accuracy
```

In [2]:
n_total   = len(df)
n_correct = (df['match'] == 'Y').sum()
n_wrong   = (df['match'] == 'N').sum()
accuracy  = n_correct / n_total

n_permitted = (df['ground_truth'] == 'PERMITTED').sum()
n_denied    = (df['ground_truth'] == 'DENIED').sum()

print('='*60)
print('  GEMMA3-4B ON HIPAA GOLDCOIN BENCHMARK')
print('='*60)
print(f'  Total cases:     {n_total}')
print(f'  Correct:         {n_correct}  ({accuracy:.1%})')
print(f'  Wrong:           {n_wrong}   ({1-accuracy:.1%})')
print()
print(f'  Label breakdown:')
print(f'    PERMITTED:     {n_permitted} cases')
print(f'    DENIED:        {n_denied} cases')
print()
print(f'  Average inference time: {df["latency_s"].mean():.1f}s per case')
print(f'  Total GPU time:         {df["latency_s"].sum()/60:.1f} minutes')
print(f'  GPU used:               V100 32GB (Stony Brook cluster)')
print('='*60)

  GEMMA3-4B ON HIPAA GOLDCOIN BENCHMARK
  Total cases:     137
  Correct:         129  (94.2%)
  Wrong:           8   (5.8%)

  Label breakdown:
    PERMITTED:     87 cases
    DENIED:        50 cases

  Average inference time: 15.8s per case
  Total GPU time:         36.0 minutes
  GPU used:               V100 32GB (Stony Brook cluster)


---
## Part 2 · One Case, Start to Finish

> **Say:** "Let's walk through one case completely — what the model saw, what it extracted, and how the engine decided the verdict."

In [3]:
# Show a correct PERMITTED case
permitted_correct = df[(df['ground_truth']=='PERMITTED') & (df['match']=='Y')].iloc[2]

print('CORRECT PREDICTION — PERMITTED case')
print('='*70)
print()
print('STEP 1 — INPUT TEXT:')
print(str(permitted_correct['question'])[:500])
print('...')
print()
print(f'STEP 2 — GROUND TRUTH: {permitted_correct["ground_truth"]}')
print(f'         CITATIONS: {permitted_correct["citations"]}')
print()

CORRECT PREDICTION — PERMITTED case

STEP 1 — INPUT TEXT:
In early 2005, a legal battle unfolded involving Anthony Camillo, a plaintiff familiar with Medicare billing practices, and Kenneth Hall Regional Hospital, the defendant. The crux of the dispute was the hospital's billing and refund records, which Camillo suspected contained evidence of fraudulent M
...

STEP 2 — GROUND TRUTH: PERMITTED
         CITATIONS: 164.502(a); 164.502(a)(1)(vi); 164.512; 164.512(e)(1)(ii)



In [4]:
# Show what the LLM extracted
sj = permitted_correct['scenario_json']
try:
    parsed = json.loads(sj) if isinstance(sj, str) else {}
except:
    parsed = {}

print('STEP 3 — EXTRACTED JSON (what Gemma3-4B extracted):')
print()

# Split into CI tuple and oracle predicates
ci_keys = ['sender','sender_role','receiver','receiver_role','subject','subject_category',
           'attribute','purpose','message_id']
oracle_keys = [k for k in parsed if k not in ci_keys and k not in ('same_org','is_business_associate')]

print('  CI 5-tuple (who sent what to whom for what purpose):')
for k in ci_keys:
    if k in parsed:
        print(f'    {k:<30} : {parsed[k]}')

print()
print('  Relationship flags:')
for k in ['same_org','is_business_associate']:
    if k in parsed:
        print(f'    {k:<30} : {parsed[k]}')

print()
print('  Oracle predicates (24 boolean flags):')
true_oracles  = [k for k in oracle_keys if parsed.get(k) == True]
false_oracles = [k for k in oracle_keys if parsed.get(k) == False]
print(f'    TRUE  ({len(true_oracles)}): {true_oracles}')
print(f'    FALSE ({len(false_oracles)}): {false_oracles[:5]} ... and {max(0,len(false_oracles)-5)} more')

STEP 3 — EXTRACTED JSON (what Gemma3-4B extracted):

  CI 5-tuple (who sent what to whom for what purpose):
    sender                         : Kenneth Hall Regional Hospital
    sender_role                    : hospital
    receiver                       : Anthony Camillo
    receiver_role                  : plaintiff
    subject                        : patients
    subject_category               : adult
    attribute                      : medical information
    purpose                        : litigation
    message_id                     : none

  Relationship flags:
    same_org                       : False
    is_business_associate          : False

  Oracle predicates (24 boolean flags):
    TRUE  (6): ['permitted_by_other_law', 'has_court_order', 'has_lawful_process_with_assurance', 'in_compliance_with_court_order', 'is_authorized_by_law_for_purpose', 'mentions_protective_order']
    FALSE (19): ['is_guardian_of_subject', 'has_authority_to_act', 'provider_patient', 'believe

In [5]:
print('STEP 4 — SOUFFLÉ ENGINE VERDICT:')
print(f'  Verdict: {permitted_correct["verdict_norm"]}')
print(f'  Match:   {permitted_correct["match"]}')
print()
print('STEP 5 — LLM EXPLANATION (what the model said):')
print(str(permitted_correct['llm_answer'])[:600])

STEP 4 — SOUFFLÉ ENGINE VERDICT:
  Verdict: PERMITTED
  Match:   Y

STEP 5 — LLM EXPLANATION (what the model said):
**VERDICT: ✅ PERMITTED**
**Answer:** The hospital’s proposed redaction of personal identifiers from the billing and refund records, as requested by Mr. Camillo, is permitted under HIPAA §164.512(e)(1)(ii). This provision allows for the disclosure of protected health information in response to a subpoena, provided there is a satisfactory assurance that the information will be used for a judicial or administrative proceeding. 

**Why:** HIPAA §164.512(e)(1)(ii) permits the disclosure of protected 


---
## Part 3 · The 8 Failures — Anatomy of LLM Errors

> **Say:** "Now the interesting part. 8 cases the model got wrong. Let's classify them and understand WHY."

In [6]:
failures = df[df['match'] == 'N'].copy().reset_index(drop=True)

# Classify error types
fp_cases = failures[failures['ground_truth'] == 'DENIED']   # predicted PERMITTED, truth DENIED
fn_cases = failures[failures['ground_truth'] == 'PERMITTED'] # predicted DENIED, truth PERMITTED

print('Error breakdown:')
print(f'  False Positives (FP): {len(fp_cases)} — said PERMITTED when truth was DENIED')
print(f'    → Safety critical: PHI could be illegally disclosed')
print(f'  False Negatives (FN): {len(fn_cases)} — said DENIED when truth was PERMITTED')
print(f'    → Over-blocking: lawful disclosure incorrectly rejected')
print()

print('All 8 failure cases:')
print()
for _, row in failures.iterrows():
    error_type = 'FP' if row['ground_truth'] == 'DENIED' else 'FN'
    print(f'  Case {row["row_id"]:>4} [{error_type}]  Truth={row["ground_truth"]:<10} Pred={row["verdict_norm"]}')
    print(f'           {str(row["question"])[:110]}...')
    print()

Error breakdown:
  False Positives (FP): 3 — said PERMITTED when truth was DENIED
    → Safety critical: PHI could be illegally disclosed
  False Negatives (FN): 5 — said DENIED when truth was PERMITTED
    → Over-blocking: lawful disclosure incorrectly rejected

All 8 failure cases:

  Case   16 [FN]  Truth=PERMITTED  Pred=DENIED
           In a small town, A.S., a young woman with a developmental disability, was under the care of her parents who we...

  Case   27 [FN]  Truth=PERMITTED  Pred=DENIED
           In a high-profile legal case, former NHL hockey players, serving as plaintiffs, are entangled in a complex leg...

  Case   30 [FN]  Truth=PERMITTED  Pred=DENIED
           Josh Sanchez, a 13-year-old juvenile, faced a significant turning point in his life after pleading guilty to a...

  Case   40 [FP]  Truth=DENIED     Pred=PERMITTED
           In 2012, an individual found himself requiring emergency medical services and was taken by ambulance to a hosp...

  Case   46 [FP]  T

In [7]:
# Deep dive into one FP and one FN
def analyze_failure(row_id, label='FP'):
    row = df[df['row_id'] == row_id].iloc[0]
    sj  = row['scenario_json']
    try:
        parsed = json.loads(sj) if isinstance(sj, str) else {}
    except:
        parsed = {}
    
    true_oracles = [k for k, v in parsed.items() if v is True]
    
    print(f'FAILURE CASE {row_id} [{label}]')
    print(f'  Ground truth: {row["ground_truth"]}')
    print(f'  LLM verdict:  {row["verdict_norm"]}')
    print(f'  Citation:     {row["citations"]}')
    print()
    print(f'  Scenario (first 300 chars):')
    print(f'  {str(row["question"])[:300]}...')
    print()
    print(f'  Oracles set TRUE by LLM: {true_oracles}')
    print()
    print(f'  LLM explanation:')
    print(f'  {str(row["llm_answer"])[:400]}')
    print('='*70)
    print()

# Show one FP and one FN
if len(fp_cases) > 0:
    analyze_failure(fp_cases.iloc[0]['row_id'], 'FP — oracle hallucination')

if len(fn_cases) > 0:
    analyze_failure(fn_cases.iloc[0]['row_id'], 'FN — under-extraction')

FAILURE CASE 40 [FP — oracle hallucination]
  Ground truth: DENIED
  LLM verdict:  PERMITTED
  Citation:     164.502(a); 164.502(a)(1)(i)

  Scenario (first 300 chars):
  In 2012, an individual found himself requiring emergency medical services and was taken by ambulance to a hospital. To receive this necessary medical attention, he provided the ambulance service, operated by the Philadelphia Fire Department Emergency Medical Services (EMS), with his personal and sen...

  Oracles set TRUE by LLM: ['is_business_associate', 'provider_patient', 'believes_unlawful_conduct', 'believes_victim_of_crime', 'is_required_by_law', 'mentions_patient_notice', 'has_treatment_relationship', 'has_ba_agreement']

  LLM explanation:
  **VERDICT: ✅ PERMITTED**
**Answer:** The use of the individual’s personal information by the Philadelphia Fire Department EMS for payment purposes, as facilitated by the contracted billing and payment processing entities, is permitted under 45 CFR §164.502(a)(1)(i), which 

In [8]:
# Which oracle predicates appear most in failure cases?
print('Oracle predicates set TRUE in FAILURE cases vs CORRECT cases:')
print()

def get_true_oracles(row):
    sj = row['scenario_json']
    try:
        parsed = json.loads(sj) if isinstance(sj, str) else {}
        ci_keys = {'sender','sender_role','receiver','receiver_role','subject',
                   'subject_category','attribute','purpose','message_id',
                   'same_org','is_business_associate'}
        return [k for k, v in parsed.items() if v is True and k not in ci_keys]
    except:
        return []

fail_oracles    = [o for _, r in failures.iterrows() for o in get_true_oracles(r)]
correct_df      = df[df['match'] == 'Y']
correct_oracles = [o for _, r in correct_df.iterrows() for o in get_true_oracles(r)]

fail_counts    = Counter(fail_oracles)
correct_counts = Counter(correct_oracles)

# Compute oracle risk: how often does this oracle appear in failures vs correct?
all_oracle_names = sorted(set(fail_counts) | set(correct_counts))

print(f"  {'Oracle predicate':<45} {'In failures':>12} {'In correct':>12}")
print('  ' + '-' * 72)
for oracle in sorted(fail_counts, key=lambda x: -fail_counts[x]):
    f_count = fail_counts.get(oracle, 0)
    c_count = correct_counts.get(oracle, 0)
    risk = ' ← HIGH RISK' if f_count >= 2 else ''
    print(f'  {oracle:<45} {f_count:>12}  {c_count:>11} {risk}')

Oracle predicates set TRUE in FAILURE cases vs CORRECT cases:

  Oracle predicate                               In failures   In correct
  ------------------------------------------------------------------------
  provider_patient                                         5           52  ← HIGH RISK
  has_treatment_relationship                               5           59  ← HIGH RISK
  has_authority_to_act                                     3           73  ← HIGH RISK
  believes_minimum_necessary                               2           24  ← HIGH RISK
  prohibited_by_other_law                                  2            3  ← HIGH RISK
  is_required_by_law                                       2           59  ← HIGH RISK
  is_authorized_by_law_for_purpose                         2           71  ← HIGH RISK
  believes_unlawful_conduct                                2           26  ← HIGH RISK
  mentions_patient_notice                                  2           36  ← HIGH RISK
  has

---
## Part 4 · Latency & Throughput Analysis

> **Say:** "While we're here — let's look at how fast the model was. This matters for production use."

In [9]:
print('Inference latency (seconds per case):')
print()
lat = df['latency_s']
stats = [
    ('Minimum',  lat.min()),
    ('25th pct', lat.quantile(0.25)),
    ('Median',   lat.median()),
    ('75th pct', lat.quantile(0.75)),
    ('Maximum',  lat.max()),
    ('Mean',     lat.mean()),
    ('Std dev',  lat.std()),
]
for label, val in stats:
    bar = '█' * int(val)
    print(f'  {label:<12} {val:>7.1f}s  {bar}')

print()
print(f'Total time for 137 cases: {lat.sum()/60:.1f} minutes')
print(f'Throughput: {3600/lat.mean():.0f} cases/hour on V100')
print()

# Does latency correlate with accuracy?
fail_lat    = df[df['match']=='N']['latency_s']
correct_lat = df[df['match']=='Y']['latency_s']
print(f'Mean latency — CORRECT cases: {correct_lat.mean():.1f}s')
print(f'Mean latency — FAILED  cases: {fail_lat.mean():.1f}s')
print()
if fail_lat.mean() > correct_lat.mean():
    print('→ Failed cases took longer — suggesting the model was more uncertain.')
else:
    print('→ No clear latency signal for failures (confidence ≠ correctness).')

Inference latency (seconds per case):

  Minimum          8.3s  ████████
  25th pct        12.5s  ████████████
  Median          14.3s  ██████████████
  75th pct        18.6s  ██████████████████
  Maximum         41.4s  █████████████████████████████████████████
  Mean            15.8s  ███████████████
  Std dev          4.6s  ████

Total time for 137 cases: 36.0 minutes
Throughput: 228 cases/hour on V100

Mean latency — CORRECT cases: 15.9s
Mean latency — FAILED  cases: 13.5s

→ No clear latency signal for failures (confidence ≠ correctness).


---
## Part 5 · Your Research — Closing the 5.8% Gap

> **Say:** "You now understand every layer of this system. The remaining 8 errors are your research target. Here's what's known about each type."

In [10]:
print('Known error patterns in the 8 failures:')
print()

error_patterns = [
    {
        'case_id': 'FP cases',
        'pattern': 'Oracle hallucination',
        'description': 'LLM sets has_court_order=True when text only implies or hedges',
        'example': '"Acting pursuant to what might be a court order"',
        'fix_direction': 'Few-shot examples with hedging negatives; stricter oracle definitions'
    },
    {
        'case_id': 'FN cases',
        'pattern': 'Under-extraction (indirect phrasing)',
        'description': 'Enabling fact is present in text but stated indirectly — LLM misses it',
        'example': '"Guardians appointed by probate court" → has_guardian but what about legal_authority?',
        'fix_direction': 'Better oracle definitions; chain-of-thought extraction prompts'
    },
    {
        'case_id': 'FN cases',
        'pattern': 'Unusual legal context',
        'description': 'Legal proceedings (NHL lawsuit, juvenile case) involve PHI disclosures the model confuses',
        'example': 'NHL players as plaintiffs in personal injury case — sport context confuses purpose extraction',
        'fix_direction': 'Domain-specific examples; explicit purpose taxonomy'
    },
]

for i, ep in enumerate(error_patterns, 1):
    print(f'  {i}. {ep["pattern"]} ({ep["case_id"]})')
    print(f'     Description:  {ep["description"]}')
    print(f'     Example:      {ep["example"]}')
    print(f'     Fix direction: {ep["fix_direction"]}')
    print()

print('='*70)
print('Research directions — choose one to pursue:')
print()
directions = [
    ('P1 Adversarial Robustness', 'Can you write new scenarios that break the model the same way?'),
    ('P2 Prompt Engineering',     'Can better oracle definitions / few-shot examples fix the FPs?'),
    ('P3 Model Comparison',       'Does a larger model (Llama3-70B) fix these same errors?'),
    ('P4 Error Explanation',      'Can you automatically diagnose WHY a prediction was wrong?'),
    ('P5 Multi-Regulation',       'Does the same error pattern appear in GDPR / CCPA?'),
    ('P6 Few-Shot Examples',      'Adding 3–5 examples to the prompt — does it close the gap?'),
]
for proj, question in directions:
    print(f'  {proj:<30} {question}')

Known error patterns in the 8 failures:

  1. Oracle hallucination (FP cases)
     Description:  LLM sets has_court_order=True when text only implies or hedges
     Example:      "Acting pursuant to what might be a court order"
     Fix direction: Few-shot examples with hedging negatives; stricter oracle definitions

  2. Under-extraction (indirect phrasing) (FN cases)
     Description:  Enabling fact is present in text but stated indirectly — LLM misses it
     Example:      "Guardians appointed by probate court" → has_guardian but what about legal_authority?
     Fix direction: Better oracle definitions; chain-of-thought extraction prompts

  3. Unusual legal context (FN cases)
     Description:  Legal proceedings (NHL lawsuit, juvenile case) involve PHI disclosures the model confuses
     Example:      NHL players as plaintiffs in personal injury case — sport context confuses purpose extraction
     Fix direction: Domain-specific examples; explicit purpose taxonomy

Research directi

In [11]:
# LIVE EXPLORATION — run this with students
# Ask: which case do you want to look at?

def show_case(row_id):
    """Full breakdown of any case by row_id."""
    rows = df[df['row_id'] == row_id]
    if len(rows) == 0:
        print(f'No case with row_id={row_id}')
        return
    row = rows.iloc[0]
    try:
        parsed = json.loads(row['scenario_json']) if isinstance(row['scenario_json'], str) else {}
    except:
        parsed = {}
    
    match_sym = '✓' if row['match'] == 'Y' else '✗'
    print(f'Case {row_id} {match_sym}  |  Truth={row["ground_truth"]}  Pred={row["verdict_norm"]}  Latency={row["latency_s"]:.1f}s')
    print()
    print('TEXT:')
    print(str(row['question'])[:600])
    print()
    print('EXTRACTED PREDICATES (TRUE only):')
    true_preds = {k: v for k, v in parsed.items() if v is True}
    for k, v in true_preds.items():
        print(f'  {k}')
    print()
    print('LLM REASONING:')
    print(str(row['llm_answer'])[:500])

# ---- TRY ANY CASE ID HERE ----
show_case(40)  # one of the FP cases

Case 40 ✗  |  Truth=DENIED  Pred=PERMITTED  Latency=12.4s

TEXT:
In 2012, an individual found himself requiring emergency medical services and was taken by ambulance to a hospital. To receive this necessary medical attention, he provided the ambulance service, operated by the Philadelphia Fire Department Emergency Medical Services (EMS), with his personal and sen

EXTRACTED PREDICATES (TRUE only):
  is_business_associate
  provider_patient
  believes_unlawful_conduct
  believes_victim_of_crime
  is_required_by_law
  mentions_patient_notice
  has_treatment_relationship
  has_ba_agreement

LLM REASONING:
**VERDICT: ✅ PERMITTED**
**Answer:** The use of the individual’s personal information by the Philadelphia Fire Department EMS for payment purposes, as facilitated by the contracted billing and payment processing entities, is permitted under 45 CFR §164.502(a)(1)(i), which allows for disclosure to the individual or their personal representative. This aligns with the covered entity’s respo

In [12]:
# Summary table: all 8 failures at a glance
print('All 8 failures at a glance:')
print()
print(f"  {'ID':>4}  {'Error':>3}  {'Truth':<10} {'Pred':<10}  {'First 80 chars of scenario'}")
print('  ' + '-' * 110)
for _, row in failures.iterrows():
    error_type = 'FP' if row['ground_truth'] == 'DENIED' else 'FN'
    snippet    = str(row['question'])[:80].replace('\n', ' ')
    print(f"  {row['row_id']:>4}  {error_type:>3}  {row['ground_truth']:<10} {row['verdict_norm']:<10}  {snippet}...")

print()
print(f'  Total errors: {len(failures)} / {len(df)} = {len(failures)/len(df):.1%}')
print(f'  Accuracy:     {n_correct} / {n_total} = {accuracy:.1%}')
print()
print('  → These 8 cases are your research targets this summer.')
print('  → Can YOU find a way to fix even one of them?')

All 8 failures at a glance:

    ID  Error  Truth      Pred        First 80 chars of scenario
  --------------------------------------------------------------------------------------------------------------
    16   FN  PERMITTED  DENIED      In a small town, A.S., a young woman with a developmental disability, was under ...
    27   FN  PERMITTED  DENIED      In a high-profile legal case, former NHL hockey players, serving as plaintiffs, ...
    30   FN  PERMITTED  DENIED      Josh Sanchez, a 13-year-old juvenile, faced a significant turning point in his l...
    40   FP  DENIED     PERMITTED   In 2012, an individual found himself requiring emergency medical services and wa...
    46   FP  DENIED     PERMITTED   In April 2011, Gary Abeyta began a therapeutic relationship with Heidi Sonntag, ...
    59   FN  PERMITTED  DENIED      In the heart of West Virginia, patient advocates from Legal Aid of West Virginia...
    66   FN  PERMITTED  DENIED      In the midst of a contentious legal b

---
## Final Summary — The Full Progression

| Approach | This week | Accuracy | What it taught us |
|---|---|---|---|
| Hand-coded rules | — | ~60% | Rules don't scale, miss synonyms |
| ML + hand features | Day 1 | ~90% | Data beats rules, but features need extraction |
| Bag of Words + ML | Day 2 | ~70% | Text → numbers works, no semantic meaning |
| TF-IDF + ML | Day 2 | ~80% | Rare words matter, still semantic gap |
| Sentence embeddings | Day 3 | ~87% | Meaning as geometry, still fails on adversarials |
| **LLM + formal engine** | **Exercise 9** | **94.2%** | **Context + scale closes most of the gap** |

> **Final say:** "94.2% is where we are. Your research question is: what gets us to 97%, 99%, 100%? You've seen every wall that led here. Now you know exactly which wall is left to climb."